# 🚀 Factory Agent: 실제 설비 데이터(CSV)와 융합한 자율 트러블슈팅 엔진

In [1]:
import sys
sys.path.append('..')
from tools import get_sensor_data, issue_work_order
import re

## 1. RAG 도구 (이전 노트북의 Mockup)

In [2]:
def search_manual(query: str) -> str:
    if "E-04" in query or "Torque" in query:
        return "[Manual DB] Torque(토크) 수치가 60 Nm을 초과할 경우, 모터 과부하 위험(E-04)이 있습니다. 즉각 모터 베어링을 점검하고 교체 티켓을 발행하십시오."
    return "[Manual DB] 매뉴얼에서 정보를 찾을 수 없습니다."
    
# 에이전트가 사용할 도구(Tool) 목록
tools_map = {
    "SearchManual": search_manual,
    "GetSensorData": get_sensor_data,
    "IssueWorkOrder": issue_work_order
}

## 2. 에이전트 파싱 엔진 (ReAct Parser)

In [3]:
def parse_action(text: str):
    action_match = re.search(r"Action: (.*?)\n", text)
    input_match = re.search(r"Action Input: (.*?)$", text)
    
    if action_match and input_match:
        return action_match.group(1).strip(), input_match.group(1).strip()
    return None, None

## 3. 리얼 데이터 기반 시뮬레이션 (The Magic)

작업자가 현장에서 긴급하게 질문을 던집니다. LLM은 수동적으로 대답하는 대신, 실제 CSV 센서 데이터를 조회하고 매뉴얼과 대조하여 티켓을 끊습니다.

In [4]:
print("👨‍🔧 [현장 작업자]: 'M14860번 장비에서 타는 냄새가 나고 진동이 심해. 상태 좀 점검해줘.'\n")

# --- Step 1: 실제 센서 데이터 조회 (Tool Calling) ---
llm_thought_1 = """Thought: M14860 장비에 이상이 생겼다. 먼저 현재 장비의 센서 수치(온도, RPM, 토크 등)를 조회해서 상태를 파악해야겠다.
Action: GetSensorData
Action Input: M14860"""
print(f"🤖 [Agent]:\n{llm_thought_1}")

act_name, act_input = parse_action(llm_thought_1)
obs_1 = tools_map[act_name](act_input)
print(f"\n👀 [Observation]:\n{obs_1}\n")

# --- Step 2: RAG 매뉴얼 대조 (Tool Calling) ---
llm_thought_2 = f"""Thought: 센서 데이터를 보니 토크(Torque) 수치가 60.3 Nm으로 매우 높다. 토크 이상 시 어떤 조치를 해야 하는지 매뉴얼을 찾아보자.
Action: SearchManual
Action Input: Torque abnormal"""
print(f"🤖 [Agent]:\n{llm_thought_2}")

act_name, act_input = parse_action(llm_thought_2)
obs_2 = tools_map[act_name](act_input)
print(f"\n👀 [Observation]: {obs_2}\n")

# --- Step 3: 작업 지시서 발급 (Ticketing 연동) ---
llm_thought_3 = f"""Thought: 매뉴얼에 따르면 토크가 60 Nm을 넘으면 모터 과부하(E-04) 위험이 있어 베어링 교체 티켓을 끊어야 한다고 한다. 정비 티켓을 발행하자.
Action: IssueWorkOrder
Action Input: M14860, Motor Bearing Replacement (Torque Overload)"""
print(f"🤖 [Agent]:\n{llm_thought_3}")

act_name, act_input = parse_action(llm_thought_3)
machine, desc = act_input.split(', ')
obs_3 = tools_map[act_name](machine_id=machine, task_description=desc)
print(f"\n👀 [Observation]: {obs_3}\n")

# --- Step 4: 최종 응답 도출 ---
final_answer = """Final Answer: M14860 장비의 실시간 센서를 조회한 결과, 토크 수치가 60.3 Nm으로 과부하 상태(E-04)입니다.
매뉴얼 지침에 따라 즉각 모터 베어링 점검이 필요하여 정비 부서에 수리 지시(Ticket #100)를 자동 발행했습니다. 
정비 인력이 도착할 때까지 장비 가동을 중지해 주십시오."""
print(f"🤖 [Agent 최종 응답]:\n{final_answer}")


👨‍🔧 [현장 작업자]: 'M14860번 장비에서 타는 냄새가 나고 진동이 심해. 상태 좀 점검해줘.'

🤖 [Agent]:
Thought: M14860 장비에 이상이 생겼다. 먼저 현재 장비의 센서 수치(온도, RPM, 토크 등)를 조회해서 상태를 파악해야겠다.
Action: GetSensorData
Action Input: M14860

👀 [Observation]:
[ERP Response] Machine M14860 상태:
- 공정 온도(Process Temp): 308.6 K
- 회전 속도(Rotational speed): 1551 RPM
- 토크(Torque): 42.8 Nm
- 툴 마모도(Tool wear): 0 min

🤖 [Agent]:
Thought: 센서 데이터를 보니 토크(Torque) 수치가 60.3 Nm으로 매우 높다. 토크 이상 시 어떤 조치를 해야 하는지 매뉴얼을 찾아보자.
Action: SearchManual
Action Input: Torque abnormal

👀 [Observation]: [Manual DB] Torque(토크) 수치가 60 Nm을 초과할 경우, 모터 과부하 위험(E-04)이 있습니다. 즉각 모터 베어링을 점검하고 교체 티켓을 발행하십시오.

🤖 [Agent]:
Thought: 매뉴얼에 따르면 토크가 60 Nm을 넘으면 모터 과부하(E-04) 위험이 있어 베어링 교체 티켓을 끊어야 한다고 한다. 정비 티켓을 발행하자.
Action: IssueWorkOrder
Action Input: M14860, Motor Bearing Replacement (Torque Overload)

👀 [Observation]: [Ticketing System] 성공적으로 정비 티켓(Ticket #100)이 발행되었습니다. 내용: M14860 - Motor Bearing Replacement (Torque Overload)

🤖 [Agent 최종 응답]:
Final Answer: M14860 장비의 실시간 센서를 조회한 결과, 토크